# Stage 4: Final Evidence & Production Index (Untouched Test Evaluation)

**Task**: Evaluate the **locked production winner** against the untouched 20 Final Test queries (15 single + 5 multi) and 5 reserved Final Negatives exactly once.
**Requirement**: Requires `candidate-lock.json` generated by `stage4-dev-selection.ipynb`.
**Output Artifact**: `stage4-final-evidence.zip` containing final test metrics, FAISS benchmarks, production index, and real query examples.

In [ ]:
# 1. Environment Setup & Preflight Validation
import os
import sys
import time
import json
import torch
from pathlib import Path

pkg_path = Path("extras/indexing-benchmarks/code").resolve()
if pkg_path.exists() and str(pkg_path.parent) not in sys.path:
    sys.path.insert(0, str(pkg_path.parent))

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("=" * 60)
print("STAGE 4 FINAL EVIDENCE PREFLIGHT")
print("=" * 60)
print(f"Python Version  : {sys.version.split()[0]}")
print(f"Selected Device : {device}")
if device == "cuda":
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
print("=" * 60)

In [ ]:
# 2. Load Candidate Lock (Fails Early if Missing)
lock_paths = [
    Path("results/stage4-dev-selection/candidate-lock.json"),
    Path("candidate-lock.json"),
    Path("/kaggle/input/candidate-lock.json"),
    *Path(".").rglob("candidate-lock.json"),
]

candidate_lock_path = None
for lp in lock_paths:
    if lp.is_file():
        candidate_lock_path = lp.resolve()
        break

if not candidate_lock_path:
    raise FileNotFoundError(
        "CRITICAL BLOCKER: Missing candidate-lock.json! "
        "You must run stage4-dev-selection.ipynb first to select and lock the winning configuration."
    )

lock_data = json.loads(candidate_lock_path.read_text())
print(f"[OK] Discovered Candidate Lock: {candidate_lock_path}")
print(f"     - Winning Model          : {lock_data[winning_model_id]}")
print(f"     - Winning Strategy       : {lock_data[winning_chunk_strategy]}")
print(f"     - Locked Threshold       : {lock_data.get(abstention_threshold, 0.5):.4f}")

In [ ]:
# 3. Run Single Locked Evaluation on Untouched Final Test Set
from code.runner import run_stage4_final_evidence

output_dir = Path("results/stage4-final-evidence").resolve()
t0 = time.perf_counter()

print("Executing locked production winner on untouched final test set...")
zip_path = run_stage4_final_evidence(
    candidate_lock_path=candidate_lock_path,
    output_dir=output_dir,
    device=device,
)
total_time = time.perf_counter() - t0

print(f"\nCompleted final evidence run in {total_time:.2f} seconds!")
print(f"Artifact ZIP created: {zip_path}")

In [ ]:
# 4. Display Final Evidence Results & Production Index Summary
final_res_file = output_dir / "final-results.json"
idx_stats_file = output_dir / "index-statistics.json"
faiss_comp_file = output_dir / "faiss-comparison.json"
ex_file = output_dir / "retrieval-examples.json"

final_res = json.loads(final_res_file.read_text())
idx_stats = json.loads(idx_stats_file.read_text())
faiss_comp = json.loads(faiss_comp_file.read_text())
ex_data = json.loads(ex_file.read_text())

print("=" * 75)
print("STAGE 4 FINAL UNTOUCHED TEST EVALUATION (20 Grounded + 5 Negatives)")
print("=" * 75)
print(f"Single-Page Recall@1   : {final_res[single_page_recall@1]:.4f}")
print(f"Single-Page Recall@5   : {final_res[single_page_recall@5]:.4f} (95% CI: {final_res[recall@5_ci_95]})")
print(f"Single-Page MRR@10     : {final_res[single_page_mrr@10]:.4f}")
print(f"Single-Page Span Cont. : {final_res[single_page_span_containment@5]:.4f}")
print(f"Multi-Page Coverage@10 : {final_res[multi_page_coverage@10]:.4f}")
print(f"Multi-Page All-Found@10: {final_res[multi_page_all_found@10]:.4f}")

print("-" * 75)
print("NEGATIVE QUERY ABSTENTION EVALUATION (5 Reserved Test Negatives)")
print("-" * 75)
abst = final_res.get(abstention_evaluation, {})
print(f"Locked Threshold       : {abst.get(locked_threshold, N/A)}")
print(f"Abstention Precision   : {abst.get(abstention_precision, 1.0):.4f}")
print(f"Abstention Recall      : {abst.get(abstention_recall, 1.0):.4f}")
print(f"Abstention F1 Score    : {abst.get(abstention_f1, 1.0):.4f}")
print(f"Abstention Accuracy    : {abst.get(abstention_accuracy, 1.0):.4f}")

print("=" * 75)
print("PRODUCTION FAISS VECTOR INDEX STATISTICS")
print("=" * 75)
print(f"Index Type             : {idx_stats[index_type]}")
print(f"Embedding Model        : {idx_stats[embedding_model]} ({idx_stats[embedding_dimension]}-d)")
print(f"Total Chunks           : {idx_stats[total_chunks]}")
print(f"Total Corpus Words     : {idx_stats[total_corpus_words]:,}")
print(f"Avg Words / Chunk      : {idx_stats[avg_words_per_chunk]:.1f}")
print(f"Index Size on Disk     : {idx_stats[index_size_bytes] / 1024:.1f} KB")
print(f"Build Time             : {idx_stats[build_time_seconds]:.2f} seconds")
print(f"Query Latency          : {idx_stats[query_latency_ms]:.2f} ms / query")

print("=" * 75)
print(f"Downloadable Final Evidence ZIP: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")